                    Phase 1 of Competitor's Strategy Analysis AI Agent

Importing API KEY from .env File

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

True

Making a Simple LLM Call

In [3]:
from huggingface_hub import InferenceClient

Client = InferenceClient(
    provider = "hf-inference",
    api_key=os.getenv("HuggingFace_API_KEY"),
    #token=os.getenv("HuggingFace_API_KEY"),
    #temperature=0.7,
)
repo_id = "HuggingFaceTB/SmolLM3-3B"

llm = Client.chat.completions.create(
    model=repo_id,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of Pakistan?",
        }
    ],
)
Response = llm.choices[0].message
print(Response)

c:\Users\aliak\OneDrive\Desktop\LangChain_Project\venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatCompletionOutputMessage(role='assistant', content="<think>\nOkay, the user is asking for the capital of Pakistan. Let me start by recalling what I know. I think the capital is Islamabad. But wait, I should make sure I'm not confusing it with another city. Sometimes people might think it's Lahore or Karachi because those are larger cities in Pakistan. Let me verify.\n\nI remember that Islamabad was established in the 1960s as the capital to replace Karachi, which was the original capital. The reason for the change was political and strategic. Karachi had issues with security and infrastructure, so they moved the capital to Islamabad. \n\nI should also check if there's any recent change or if there's a possibility that the capital has changed. But as far as I know, Islamabad has been the capital since 1960. There's a possibility that the government might consider moving the capital again, but that's not something that's happened recently. \n\nAnother thing to consider is the administ

Loading Files(PDF, TXT, and CSV ) & URLs Via Document Loader

In [4]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader, UnstructuredURLLoader
import os

data_file = "data"

documents = []

# Loop through files in the data folder
for file in os.listdir(data_file):
    try:
        file_path = os.path.join(data_file, file)

        if file.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
        elif file.endswith(".txt"):
            loader = TextLoader(file_path, encoding="utf-8")  # encoding avoids errors
        elif file.endswith(".csv"):
            loader = CSVLoader(file_path, source_column="industry")  # You can add delimiter="," if needed
        else:
            print(f"Skipped {file}, can't upload this file.")
            continue

        # Load data and extend the documents list
        data = loader.load()
        documents.extend(data)
        print(f"Loaded {len(data)} documents from {file}")

    except Exception as e:
        print(f"Error loading {file}: {e}")

# Load from URL
try:
    url = "https://www.digierapro.com"
    loader = UnstructuredURLLoader(urls=[url])
    data = loader.load()
    documents.extend(data)
    print(f"Loaded {len(data)} documents from URL: {url}")
except Exception as e:
    print(f"Error loading URL {url}: {e}")

print(f"\nTotal documents loaded: {len(documents)}")


Loaded 12 documents from DigitalRightsFoundation.pdf
Error loading Free List of Marketing and advertising Businesses export 2025-08-09 14-46-29.csv: Error loading data\Free List of Marketing and advertising Businesses export 2025-08-09 14-46-29.csv
Loaded 41 documents from National AI Policy Consultation Draft V1.pdf
Loaded 1 documents from Tech_Stack.txt
Loaded 1 documents from URL: https://www.digierapro.com

Total documents loaded: 55


Adding Google Search Tool to get data by searching through websites on Google

In [5]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv("Google_API_KEY")
os.environ["GOOGLE_CSE_ID"] = os.getenv("Google_CSE_ID")

from langchain_core.tools import Tool
from langchain_google_community import GoogleSearchAPIWrapper

search = GoogleSearchAPIWrapper()

tool = Tool(
    name="Google Search",
    description="Search Google for recent results.",
    func=search.run,
)
query = input(str("What do You want to search from Google? "))

data = tool.run(query)
print(f"Google Search Result: {data}")

#Adding this data into the documents list as well as did in above cell
documents.extend(data)
print(f"Loaded {len(data)} documents from Google Search")
print(f"\nTotal documents after Google Search: {len(documents)}")

Google Search Result: Pakistan's prime minister leads the executive branch of the federal government, oversees the state economy, leads the National Assembly, heads the Council of ... Prime Minister Muhammad Shehbaz Sharif, President of Turkiye Recep Tayyip Erdogan and President of Azerbaijan Ilham Aliyev holding hands in solidarity showing ... Mian Muhammad Shehbaz Sharif (born 23 September 1951) is a Pakistani politician and businessman who has served as the 20th prime minister of Pakistan since ... Apr 27, 2025 ... Wang Yi Has a Phone Call with Pakistani Deputy Prime Minister and Foreign Minister Mohammad Ishaq Dar. Apr 30, 2025 ... Today, Secretary Marco Rubio spoke with Prime Minister of Pakistan Muhammad Shehbaz Sharif. The Secretary spoke of the need to condemn the ... Jul 18, 2025 ... Below is a list of prime ministers of Pakistan since its independence. A name in italics indicates a caretaker prime minister. May 8, 2025 ... Secretary Marco Rubio spoke today with Pakistani Prim

Splitting Files into Chunks

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=45,
    length_function=len,
    add_start_index=True,
)

from langchain.schema import Document

# Convert each string in documents list to a Document object because text_splitter.split_documents expects a list of Document objects
# If documents are already Document objects, this step is not necessary.
documents_objs = [Document(page_content=doc) if isinstance(doc, str) else doc for doc in documents]

#It uses the split_documents method to split the loaded documents into smaller chunks based on the specified parameters.
chunks = text_splitter.split_documents(documents_objs)
print(len(chunks), "chunks created")

2279 chunks created


Creating Memory for Agent to Remember Conversation

In [7]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history", 
    return_messages=True,
    input_key="input",
    output_key="output",
    human_prefix="Human",
    ai_prefix="AI",
    max_token_limit=1000,  # Adjust this limit based on your needs
    # This will limit the memory to the last 1000 tokens, you can adjust this as needed
)
# Example usage of memory
memory.save_context({"input": "Hello"}, {"output": "Hi there!"})    
# Retrieve the chat history
chat_history = memory.load_memory_variables({})["chat_history"]
print("Chat History:", chat_history)
# Note: The above code is a simplified example of how to use the ConversationBufferMemory.
# In practice, you would integrate this memory with your agent or LLM to maintain context across
# interactions.


Chat History: [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi there!', additional_kwargs={}, response_metadata={})]


C:\Users\aliak\AppData\Local\Temp\ipykernel_15624\990486328.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


Converting Chunks into Embeddings using HuggingFaceEmbeddings 

In [9]:

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True}

)


Assembling these Embeddings and Storing in Vector DB(FAISS or Chroma DB)

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = None
for i in range(0, len(documents), 50):  # process in chunks of 50 docs
    chunk_store = FAISS.from_documents(documents[i:i+50], embeddings)
    if vectorstore:
        vectorstore.merge_from(chunk_store)
    else:
        vectorstore = chunk_store
#To prevent MemoryError, we process the documents in smaller chunks of 50 at a time.

Unexpected exception formatting exception. Falling back to standard exception
Unexpected exception formatting exception. Falling back to standard exception
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\aliak\OneDrive\Desktop\LangChain_Project\venv2\Lib\site-packages\IPython\core\interactiveshell.py", line 2194, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
        etype, value, tb, tb_offset=tb_offset
    )
  File "c:\Users\aliak\OneDrive\Desktop\LangChain_Project\venv2\Lib\site-packages\IPython\core\ultratb.py", line 1182, in structured_traceback
    return FormattedTB.structured_traceback(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        self, etype, evalue, etb, tb_offset, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\aliak\OneDrive\Desktop\LangChain_Project\venv2\Lib\site-packages\IPython\core\ultratb.py", line 1053, in structured_traceback
    return VerboseTB.structured_traceback(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        self, etype, evalue, etb, tb_offset, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\al

: 

Linking RAG PipeLine with QARetrieval & HuggingFacePipeline with Chains

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

# 1. Load tokenizer and model
model_id = "Qwen/Qwen2-1.5B-Instruc"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# 2. Create HF pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.7,
    max_new_tokens=512
)

# 3. Wrap pipeline for LangChain
hf_pipeline = HuggingFacePipeline(pipeline=pipe)

# 4. Wrap in ChatHuggingFace
chat = ChatHuggingFace(llm=hf_pipeline)

# 5. Test
response = chat.invoke("What is AI Policy of Pakistan focused on and what are digital right of citizens in pakistan?")
print(response.content)


c:\Users\aliak\OneDrive\Desktop\LangChain_Project\venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: Qwen/Qwen2-1.5B-Instruc is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

Tools and Agent Executioner

In [1]:
from langchain.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage, AIMessage

tools = [
    Tool(
        name="Google Search",
        description="Search Google for recent results.",
        func=search.run,
    ),
    Tool(
        name="HuggingFace LLM",
        description="Use the HuggingFace LLM to answer questions.",
        func=lambda query: chat.invoke(query).content,
    ),
    Tool(
        name="Vector Store Search",
        description="Search the vector store for relevant documents.",
        func=lambda query: vectorstore.similarity_search(query, k=5),
    ),

    Tool(

        name="Memory",
        description="Retrieve the chat history from memory.",           
        func=lambda _: memory.load_memory_variables({})["chat_history"],
    ),
]
agent = initialize_agent(
    tools=tools,
    llm=chat,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
)
query = "What is the capital of Pakistan?"      
response = agent.invoke(
    {
        "input": query,
        "chat_history": memory.load_memory_variables({})["chat_history"],
    }
)
print(f"Agent Response: {response['output']}")


NameError: name 'search' is not defined

The code you posted does **not** have an indentation error in the shown cell. However, if you are seeing an "Expected indented block" error, it usually means that somewhere in your notebook, a function, loop, or conditional statement is missing an indented body.

**Common causes:**
- A function or loop is declared but has no code inside, or the code is not indented.
- A cell was run with incomplete code.

**How to fix:**
1. **Check for missing indentation** after statements like `for`, `if`, `def`, `class`, etc.
2. **Example of correct indentation:**
    ````python
    for i in range(5):
        print(i)  # This line is indented
    ````

3. **If you want an empty block**, use `pass`:
    ````python
    for i in range(5):
        pass
    ````

**In your notebook:**  
- Review all code cells above and below the error for missing or incorrect indentation.
- If you recently edited a cell, make sure all blocks are properly indented.

If you share the exact error message and the cell where it occurs, I can give a more specific fix.

The code you posted does **not** have an indentation error in the shown cell. However, if you are seeing an "Expected indented block" error, it usually means that somewhere in your notebook, a function, loop, or conditional statement is missing an indented body.

**Common causes:**
- A function or loop is declared but has no code inside, or the code is not indented.
- A cell was run with incomplete code.

**How to fix:**
1. **Check for missing indentation** after statements like `for`, `if`, `def`, `class`, etc.
2. **Example of correct indentation:**
    ````python
    for i in range(5):
        print(i)  # This line is indented
    ````

3. **If you want an empty block**, use `pass`:
    ````python
    for i in range(5):
        pass
    ````

**In your notebook:**  
- Review all code cells above and below the error for missing or incorrect indentation.
- If you recently edited a cell, make sure all blocks are properly indented.

If you share the exact error message and the cell where it occurs, I can give a more specific fix.